# Energy Consumption Prediction: Exploration & Local Testing

Use this notebook for **data exploration** and **local model testing**.
Changes here do **not** affect your submission — when you are happy with your
model, copy the `EnergyConsumptionModel` class back to the **submission notebook**.

## Workflow
1. Explore the data below.
2. Iterate on your model in this notebook using a train/test split (train: before 2025-12-01, test: Dec 2025 – Feb 2026).
3. When satisfied, copy your `EnergyConsumptionModel` class (and its imports) to the submission notebook.
4. Run the Submit cell in the submission notebook.

In [0]:
!pip install autogluon==1.5.0

In [0]:
!nvidia-smi

## Imports

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F, Window

## Load Data

Row-level security ensures you only see data **up to end of November 2025** interactively.

**Tip:** The dataset is large. Keep data as PySpark DataFrames. If you want to
explore in pandas, use `df.limit(N).toPandas()` on a small sample.

In [0]:
df = spark.table("datathon.shared.client_consumption")
display(df)

In [0]:
# ---------------------------------------------------------------
# Aggregate 182M raw rows -> ~32K rows (one per 15-min interval)
# Single national series. Target is per-client load; we rescale by n_clients
# when computing the Spain-total MAE.
# ---------------------------------------------------------------
agg_df = (
    df
    .withColumn("datetime_15min", F.window("datetime_local", "15 minutes").start)
    .groupBy("datetime_15min")
    .agg(
        F.sum("active_kw").alias("active_kw"),
        F.countDistinct("client_id").alias("n_clients"),
    )
    .orderBy("datetime_15min")
)

ts_pdf = agg_df.toPandas()
ts_pdf["datetime_15min"] = pd.to_datetime(ts_pdf["datetime_15min"])
ts_pdf = ts_pdf.sort_values("datetime_15min").reset_index(drop=True)
ts_pdf["item_id"] = "Total_Grid"
ts_pdf["avg_kw_per_client"] = ts_pdf["active_kw"] / ts_pdf["n_clients"]

print(f"Shape: {ts_pdf.shape}")
print(f"Date range: {ts_pdf['datetime_15min'].min()} -> {ts_pdf['datetime_15min'].max()}")
print(f"Memory: {ts_pdf.memory_usage(deep=True).sum() / 1e6:.1f} MB")
display(ts_pdf.head(10))

In [0]:
import mlflow
import time
import random

# ---------------------------------------------------------------
# Random run name generator (adjective-noun, like Docker containers).
# Used to name each MLflow run the experiment starts below.
# ---------------------------------------------------------------
_ADJECTIVES = [
    "bold", "calm", "dark", "eager", "fair", "glad", "keen", "loud",
    "mild", "neat", "odd", "pale", "quick", "rare", "slim", "tall",
    "vast", "warm", "zany", "bright", "crisp", "dense", "fiery",
    "grand", "hardy", "icy", "jolly", "lush", "noble", "proud",
    "rapid", "sharp", "swift", "vivid", "witty", "agile", "brisk",
]
_NOUNS = [
    "aurora", "blaze", "comet", "delta", "ember", "frost", "gale",
    "haze", "iris", "jade", "kite", "lark", "moon", "nova", "opal",
    "peak", "quasar", "ridge", "spark", "tide", "vale", "wave",
    "zenith", "bolt", "cedar", "dawn", "flare", "grove", "hawk",
    "orbit", "pulse", "reef", "storm", "torch", "vortex", "zephyr",
]

def random_run_name():
    return f"{random.choice(_ADJECTIVES)}-{random.choice(_NOUNS)}-{random.randint(10, 99)}"

# Shared run-name prefix used as MODEL_DIR suffix below
RUN_NAME = random_run_name()
print(f"Base run name: {RUN_NAME}")

# Safety: make sure no stale run is active before the experiment loop
mlflow.end_run()

In [0]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# ---------------------------------------------------------------
# Per-region-group AutoGluon runs (one independent predictor each).
# Predictions are aggregated (summed) to Spain total after rescaling by n_clients.
# ---------------------------------------------------------------
MODEL_DIR         = f"/Workspace/datathon/pingpong_club_epoch/AutogluonModels/{RUN_NAME}"
TARGET            = "avg_kw_per_client"
PREDICTION_LENGTH = 96     # 1 day at 15-min freq
TIME_LIMIT        = 120    # per region

print(f"Models saved under: {MODEL_DIR}")
print(f"Target:             {TARGET}")
print(f"Regions:            {sorted(ts_pdf['item_id'].unique())}")
print(f"Time limit:         {TIME_LIMIT}s per region "
      f"({TIME_LIMIT * ts_pdf['item_id'].nunique()}s total)")

## Target-transform comparison — leaderboard-based Spain MAE proxy

Six approaches. All use the same fitting setup; they differ only in what the target column is.

**Residuals (per-client minus lag — translation-invariant MAE):**
- **A1 residual 48h**: `T - T_lag_48h`
- **A2 residual 24h**: `T - T_lag_24h`
- **A3 residual weekly**: `T - T_lag_weekly`
- **A4 residual daily+weekly avg**: `T - (T_lag_24h + T_lag_weekly) / 2`

**Baselines (no residual subtraction):**
- **A5 raw active_kw**: predict Spain total directly (no per-client normalization).
- **A6 per_client**: predict `avg_kw_per_client` (= base `autogluon` notebook target).

**Spain MAE proxy** = `abs(leaderboard.best_score_val) × rescale_factor`
- `rescale_factor = n_clients[-1]` for per-client targets (A1–A4, A6).
- `rescale_factor = 1` for A5 (already in total kW).

This gives a fair same-scale comparison across all six, using AutoGluon's internal 5-window `score_val` on its `WeightedEnsemble`. One MLflow run per approach (6 total).

In [0]:
# ---------------------------------------------------------------
# Lag features + 6 target-transform approaches:
#   A1-A4 : residuals  (per_client - lag)              rescale: × n_clients[-1]
#   A5    : raw Spain active_kw                         rescale: none (already kW)
#   A6    : per_client only (no residual subtraction)   rescale: × n_clients[-1]
#
# AutoGluon's internal num_val_windows bumped to 10 for more robust model selection.
# ---------------------------------------------------------------
LAG_24H            = 96
LAG_48H            = 192
LAG_WEEK           = 672
NUM_VAL_WINDOWS    = 10    # AutoGluon-internal rolling validation windows

ts_pdf_lag = ts_pdf.copy().sort_values(["item_id", "datetime_15min"])
grp = ts_pdf_lag.groupby("item_id", group_keys=False)
ts_pdf_lag["t_lag_24h"]  = grp["avg_kw_per_client"].shift(LAG_24H)
ts_pdf_lag["t_lag_48h"]  = grp["avg_kw_per_client"].shift(LAG_48H)
ts_pdf_lag["t_lag_week"] = grp["avg_kw_per_client"].shift(LAG_WEEK)
ts_pdf_lag["t_lag_dw"]   = (ts_pdf_lag["t_lag_24h"] + ts_pdf_lag["t_lag_week"]) / 2

ts_pdf_lag["target_A1_res_48h"]    = ts_pdf_lag["avg_kw_per_client"] - ts_pdf_lag["t_lag_48h"]
ts_pdf_lag["target_A2_res_24h"]    = ts_pdf_lag["avg_kw_per_client"] - ts_pdf_lag["t_lag_24h"]
ts_pdf_lag["target_A3_res_week"]   = ts_pdf_lag["avg_kw_per_client"] - ts_pdf_lag["t_lag_week"]
ts_pdf_lag["target_A4_res_dw"]     = ts_pdf_lag["avg_kw_per_client"] - ts_pdf_lag["t_lag_dw"]
ts_pdf_lag["target_A5_raw"]        = ts_pdf_lag["active_kw"]
ts_pdf_lag["target_A6_per_client"] = ts_pdf_lag["avg_kw_per_client"]

APPROACHES = {
    "A1_residual_48h":    {"target": "target_A1_res_48h",    "requires": ["t_lag_48h"],  "rescale_by_n": True},
    "A2_residual_24h":    {"target": "target_A2_res_24h",    "requires": ["t_lag_24h"],  "rescale_by_n": True},
    "A3_residual_weekly": {"target": "target_A3_res_week",   "requires": ["t_lag_week"], "rescale_by_n": True},
    "A4_residual_dw_avg": {"target": "target_A4_res_dw",     "requires": ["t_lag_dw"],   "rescale_by_n": True},
    "A5_raw_active_kw":   {"target": "target_A5_raw",        "requires": [],             "rescale_by_n": False},
    "A6_per_client":      {"target": "target_A6_per_client", "requires": [],             "rescale_by_n": True},
}


def run_approach(name, cfg):
    """Train per-region on cfg['target']; Spain MAE via leaderboard score_val proxy."""
    target_col   = cfg["target"]
    requires     = cfg["requires"]
    rescale_by_n = cfg["rescale_by_n"]

    df = ts_pdf_lag.dropna(subset=[target_col] + requires).copy()

    per_region = []

    for region_id in sorted(df["item_id"].unique()):
        region_df = df[df["item_id"] == region_id][
            ["item_id", "datetime_15min", target_col]
        ]
        td = TimeSeriesDataFrame.from_data_frame(
            region_df, id_column="item_id", timestamp_column="datetime_15min",
        )
        predictor = TimeSeriesPredictor(
            path=f"{MODEL_DIR}/{name}/{region_id}",
            prediction_length=PREDICTION_LENGTH,
            target=target_col,
            eval_metric="MAE",
            freq="15min",
        )
        predictor.fit(
            td,
            presets="best_quality",
            verbosity=1,
            time_limit=TIME_LIMIT,
            excluded_model_types=["TemporalFusionTransformer"],
            num_val_windows=NUM_VAL_WINDOWS,
        )

        lb = predictor.leaderboard()
        best_score_val = abs(float(lb.iloc[0]["score_val"]))
        best_model     = str(lb.iloc[0]["model"])

        n_clients_last = int(
            ts_pdf[ts_pdf["item_id"] == region_id]["n_clients"].iloc[-1]
        )
        scale_factor  = n_clients_last if rescale_by_n else 1
        spain_proxy   = best_score_val * scale_factor

        per_region.append({
            "region_id":        region_id,
            "best_model":       best_model,
            "score_val":        best_score_val,
            "n_clients_last":   n_clients_last,
            "rescale_by_n":     rescale_by_n,
            "spain_mae_proxy":  spain_proxy,
        })

    spain_total_mae = float(sum(r["spain_mae_proxy"] for r in per_region))

    return {
        "per_region":      per_region,
        "spain_total_mae": spain_total_mae,
    }

In [0]:
# ---------------------------------------------------------------
# One MLflow run per approach (6 total).
# spain_total_mae = abs(leaderboard_best_score_val) * rescale_factor
#   rescale_factor = n_clients[-1] when predicting a per-client target, else 1.
# ---------------------------------------------------------------
mlflow.end_run()

approach_results = {}

for name, cfg in APPROACHES.items():
    run_name = f"{random_run_name()}-{name}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "experiment_type":       "target_transform_scoreval_proxy",
            "approach":              name,
            "target_col":            cfg["target"],
            "rescale_by_n":          cfg["rescale_by_n"],
            "prediction_length":     PREDICTION_LENGTH,
            "time_limit_per_region": TIME_LIMIT,
            "num_val_windows":       NUM_VAL_WINDOWS,
            "n_regions":             int(ts_pdf["item_id"].nunique()),
        })

        print(f"\n=== {name}  ({run_name}) ===")
        r = run_approach(name, cfg)
        approach_results[name] = r

        mlflow.log_metric("spain_total_mae", r["spain_total_mae"])
        for row in r["per_region"]:
            rid = row["region_id"]
            mlflow.log_metric(f"score_val_{rid}",       row["score_val"])
            mlflow.log_metric(f"n_clients_last_{rid}",  row["n_clients_last"])
            mlflow.log_metric(f"spain_mae_proxy_{rid}", row["spain_mae_proxy"])
            mlflow.set_tag(f"best_model_{rid}",         row["best_model"])

        print(f"  spain_total_mae (proxy): {r['spain_total_mae']:,.0f}")

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------
rows = []
for name, r in approach_results.items():
    first = r["per_region"][0]
    rows.append({
        "approach":        name,
        "best_model":      first["best_model"],
        "score_val":       first["score_val"],
        "rescale_by_n":    first["rescale_by_n"],
        "n_clients_last":  first["n_clients_last"],
        "spain_total_mae": r["spain_total_mae"],
    })
summary = pd.DataFrame(rows).sort_values("spain_total_mae")
print("\nSummary (sorted by spain_total_mae):")
display(summary)